# KL-IG: 9-Method Evaluation (3σ variants + 5 baselines)
ResNet50, 100 images, 7 core metrics

#Setup

In [16]:
!pip install nbconvert
!jupyter nbconvert --clear-output --inplace "/content/drive/MyDrive/Colab Notebooks/evaluation_main_notebook.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/evaluation_main_notebook.ipynb to notebook
[NbConvertApp] Writing 125566 bytes to /content/drive/MyDrive/Colab Notebooks/evaluation_main_notebook.ipynb


In [ ]:
!git clone https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
!pip install -q captum datasets opencv-python-headless


In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.auto import tqdm
import cv2

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "infocube-main",
                  Path("/content/KLIG_V1/infocube-main"),
                  Path("/content/KLIG_V1")]:
    if (candidate / "klig").exists():
        ROOT = candidate
        break
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.compare.captum_baselines import (
    run_ig, run_smoothgrad, run_expected_gradients, _absmax_collapse,
)
from captum.attr import IntegratedGradients, Saliency

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")


In [ ]:
#SAMPLE SIZE
N_IMGS    = 1000
N_subset  = 100
N_STEPS   = 50
N_SAMPLES = 10
TIER_A    = 1000
BLUR_SIGMA  = 16.0
BLUR_KERNEL = 51
IG_STEPS  = 50
EG_SAMPLES = 50
SG_SAMPLES = 50
BIG_STEPS  = 50
BIG_SIGMA  = 10.0

N_INSERTION_STEPS   = 50
N_SENS_SUBSETS      = 30
SENS_FRACTIONS      = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8]
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
PERTURBATION_RUNS   = 3
OCCLUSION_PATCH  = 14
OCCLUSION_STRIDE = 7
OCCLUSION_RATIOS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

HF_DATASET_NAME = "evanarlian/imagenet_1k_resized_256"
HF_SPLIT = "val"

# ── Persistent cache ─────────────────────────────────────

USE_DRIVE       = True
DRIVE_CACHE     = "/content/drive/MyDrive/klig_eval_cache"
LOCAL_CACHE     = "eval_cache"
FORCE_RECOMPUTE = False
if USE_DRIVE:
    try:
        from google.colab import drive
        if not Path("/content/drive").exists() or not any(Path("/content/drive").iterdir()):
            drive.mount("/content/drive")
        CACHE_DIR = Path(DRIVE_CACHE)
        print(f"[cache] Drive: {CACHE_DIR}")
    except Exception as e:
        CACHE_DIR = Path(LOCAL_CACHE)
        print(f"[cache] Drive unavailable ({e}) — falling back to {CACHE_DIR}")
else:
    CACHE_DIR = Path(LOCAL_CACHE)
    print(f"[cache] Local: {CACHE_DIR}")

CACHE_DIR.mkdir(parents=True, exist_ok=True)


methods_all = [
    "KL-IG (adaptive)",
    "KL-IG (σ=0.25)",
    "IDG",
    "ExpGrad",
    "IG-zero",
    "SmoothGrad",
    "Vanilla Grad",
    "Blur-IG",
]

COLORS_ALL = {
    "KL-IG (adaptive)":  "#1B5E3F",
    "KL-IG (σ=0.25)": "#3CB371",
    "IDG":            "#E07B39",
    "ExpGrad":        "#DC143C",
    "IG-zero":        "#7B68EE",
    "SmoothGrad":     "#1E90FF",
    "Vanilla Grad":   "#8B4513",
    "Blur-IG":        "#20B2AA",
}

print(f"N_IMGS={N_IMGS}, methods={len(methods_all)}")

In [ ]:
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

def make_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    k1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    k1d = k1d / k1d.sum()
    kh = k1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
    kw = k1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
    pad = kernel_size // 2
    out = F.conv2d(x, kh, padding=(pad, 0), groups=3)
    return F.conv2d(out, kw, padding=(0, pad), groups=3)

def make_eg_background(x, n=50):
    return torch.randn(n, *x.squeeze(0).shape, device=x.device)

model, imagenet_labels = load_model()
print(f"Model: ResNet50, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


In [ ]:
#from transformers import AutoImageProcessor, AutoModelForImageClassification
#import torch
#import torch.nn.functional as F
#
#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
#
#class DeiTWrapper(torch.nn.Module):
#
#    def __init__(self, model):
#        super().__init__()
#        self.model = model
#    def forward(self, x):
#        return self.model(pixel_values=x).logits
#
#def load_model():
#    model_name = "facebook/deit-small-patch16-224"
#    processor  = AutoImageProcessor.from_pretrained(model_name)
#    base       = AutoModelForImageClassification.from_pretrained(model_name)
#    model      = DeiTWrapper(base).to(DEVICE).eval()
#    categories = [base.config.id2label[i] for i in range(base.config.num_labels)]
#    return model, processor, categories
#
#def denormalize(x):
#    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
#    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(-1, 1, 1)
#    if x.dim() == 4:
#        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
#    return (x * std + mean).clamp(0, 1)
#
#def make_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
#    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
#    k1d = torch.exp(-0.5 * (coords / sigma) ** 2)
#    k1d = k1d / k1d.sum()
#    kh  = k1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
#    kw  = k1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
#    pad = kernel_size // 2
#    out = F.conv2d(x, kh, padding=(pad, 0), groups=3)
#    return F.conv2d(out, kw, padding=(0, pad), groups=3)
#
#def make_eg_background(x, n=50):
#    return torch.randn(n, *x.squeeze(0).shape, device=x.device)
#
#model, processor, imagenet_labels = load_model()
#print(f"DeiT: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

In [ ]:
def load_imagenet_subset(n_images):
    try:
        from datasets import load_dataset
        print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}]")
        ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True) \
         .shuffle(seed=42, buffer_size=50_000)
        seen, out = set(), []
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break
        return out
    except Exception as e:
        print(f"[dataset] Failed: {e}")
        return []

raw_samples = load_imagenet_subset(TIER_A)
dataset = []
with torch.no_grad():
    for i, (pil, gt) in enumerate(tqdm(raw_samples, desc="prep")):
        x = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top1 = int(probs.argmax())
        target = gt if probs[gt].item() > 0.05 else top1
        dataset.append({
            "idx": i,
            "x": x,
            "target": target,
            "label_str": imagenet_labels[target],
        })

results = {}
for row in dataset:
    results[row["idx"]] = row

eval_idx = list(range(min(N_IMGS, len(dataset))))
print(f"Dataset: {len(dataset)} images, evaluating {len(eval_idx)}")


In [ ]:
##Catogery just to get wide range of samples while experimenting for smaller subset
CATEGORY_RANGES = {
    "fish":       [(0, 6), (389, 397)],
    "bird":       [(7, 24), (80, 100)],
    "reptile":    [(33, 68)],
    "dog":        [(151, 268)],
    "cat":        [(281, 285), (291, 293)],
    "bear":       [(294, 297)],
    "primate":    [(365, 382)],
    "instrument": [(401, 420), (486, 546), (683, 776)],
    "vehicle":    [(403, 408), (436, 436), (468, 468),
                   (555, 555), (627, 627), (654, 654),
                   (656, 656), (675, 675), (717, 717),
                   (734, 734), (779, 779), (817, 817),
                   (864, 864), (867, 867), (874, 874)],
    "structure":  [(425, 425), (483, 483), (497, 497),
                   (562, 562), (668, 668), (698, 698),
                   (727, 727), (832, 832), (975, 975)],
    "object":     [(409, 410), (440, 442), (487, 487),
                   (508, 508), (530, 530), (587, 587),
                   (664, 664), (754, 754), (782, 782)],
    "food":       [(924, 969)],
    "nature":     [(970, 999)],
}

def in_any_range(y, ranges):
    return any(lo <= y <= hi for lo, hi in ranges)

def load_imagenet_subset(n_images):
    from datasets import load_dataset
    print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}] (stratified)")
    ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)

    per_cat = max(1, n_images // len(CATEGORY_RANGES))
    bucket  = {cat: 0 for cat in CATEGORY_RANGES}
    seen, out = set(), []

    for ex in ds:
        y = int(ex["label"])
        if y in seen:
            continue
        for cat, ranges in CATEGORY_RANGES.items():
            if bucket[cat] < per_cat and in_any_range(y, ranges):
                seen.add(y)
                bucket[cat] += 1
                out.append((ex["image"].convert("RGB"), y))
                break
        if len(out) >= n_images:
            break

    if len(out) < n_images:
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break

    print(f"[dataset] per-category counts: {bucket}")
    return out

In [ ]:
def get_sigma_final(sigma_name, model, x, target):
    if sigma_name == "adaptive":
        # Adaptive σ_stop, capped at 1.0
        return min(max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0), 1.0)
    elif sigma_name == "σ=0.25":
        return 0.25
    else:
        raise ValueError(f"Unknown sigma variant: {sigma_name!r}")


def reduce_baseline_attr(attr: torch.Tensor) -> torch.Tensor:
    """Sum-collapse (signed) for all metrics except Gini."""
    if attr.dim() == 4:
        attr = attr.squeeze(0)
    return attr.sum(dim=0)

def compute_klig(model, x, target, sigma_final):
    attr = ImageAttributor(model=model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                           sigma_final=sigma_final, device=DEVICE)
    res = attr.attribute(x, target=target, show_progress=False)
    return res, res.attr_map("sum")

def compute_ig_zero(model, x, target):
    return run_ig(model, x, target, n_steps=IG_STEPS)

def compute_idg(model, x, target):
    x_inp = x.clone().detach().requires_grad_(True)
    model.zero_grad()
    out = model(x_inp)
    out[0, target].backward()
    grad = x_inp.grad
    if grad is None:
        return torch.zeros(x.shape[-2:], device=x.device)
    attr = x_inp.detach() * grad.detach()
    return reduce_baseline_attr(attr)

def compute_expgrad(model, x, target):
    bg = make_eg_background(x, n=EG_SAMPLES)
    return run_expected_gradients(model, x, target, background=bg, n_samples=EG_SAMPLES)

def compute_smoothgrad(model, x, target):
    return run_smoothgrad(model, x, target, n_samples=SG_SAMPLES)

def compute_vanilla_grad(model, x, target):
    sal = Saliency(model)
    attr = sal.attribute(x, target=target, abs=False)
    return reduce_baseline_attr(attr.detach())

def compute_blurig(model, x, target, n_steps=BIG_STEPS, sigma_max=BIG_SIGMA):
    """Blur-IG: Xu et al. CVPR 2020."""
    from scipy.ndimage import gaussian_filter as sp_gf
    model.eval()
    x_np = x[0].detach().cpu().numpy()
    x_baseline_np = np.stack([sp_gf(x_np[c], sigma=sigma_max) for c in range(x_np.shape[0])])
    x_baseline = torch.tensor(x_baseline_np, dtype=x.dtype, device=x.device)
    integrated = torch.zeros_like(x[0])
    for k in range(n_steps):
        alpha = (k + 0.5) / n_steps
        sigma_k = sigma_max * (1.0 - alpha)
        if sigma_k > 0.01:
            xb_np = np.stack([sp_gf(x_np[c], sigma=sigma_k) for c in range(x_np.shape[0])])
        else:
            xb_np = x_np.copy()
        xb = torch.tensor(xb_np, dtype=x.dtype, device=x.device).unsqueeze(0)
        xb.requires_grad_(True)
        model.zero_grad()
        model(xb)[0, target].backward()
        if xb.grad is not None:
            integrated += xb.grad[0].detach()
    attr = (x[0] - x_baseline) * integrated / n_steps
    return reduce_baseline_attr(attr.detach())

COMPUTE_FN = {
    "IDG":          compute_idg,
    "ExpGrad":      compute_expgrad,
    "IG-zero":      compute_ig_zero,
    "SmoothGrad":   compute_smoothgrad,
    "Vanilla Grad": compute_vanilla_grad,
    "Blur-IG":      compute_blurig,
}

print("Attribution functions ready")

In [ ]:
# ── Collapse helpers ──
def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0, keepdim=True)
    return a.gather(0, idx).squeeze(0)

def sum_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    return a.sum(dim=0)

def _xb(model, x):
    xb = x.unsqueeze(0) if x.dim() == 3 else x
    return xb.to(next(model.parameters()).device)

# ── Raw (C,H,W) attribution helpers ──
from captum.attr import IntegratedGradients, Saliency

def raw_klig(model, x, target, sigma_final):
    return ImageAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                           sigma_final=sigma_final).attribute(x, target=target).attr

def raw_ig_zero(model, x, target):
    xb = _xb(model, x)
    return IntegratedGradients(model).attribute(
        xb, baselines=torch.zeros_like(xb), target=target,
        n_steps=IG_STEPS, method="gausslegendre",
        internal_batch_size=IG_STEPS).detach().squeeze(0)

def raw_blur_ig(model, x, target):
    xb = _xb(model, x)
    blur = make_blur_baseline(x)
    if blur.dim() == 3: blur = blur.unsqueeze(0)
    return IntegratedGradients(model).attribute(
        xb, baselines=blur.to(xb.device), target=target,
        n_steps=IG_STEPS, method="gausslegendre",
        internal_batch_size=IG_STEPS).detach().squeeze(0)

def raw_smoothgrad(model, x, target):
    xb  = _xb(model, x)
    std = 0.15 * float((xb.max() - xb.min()).item())
    noisy = (xb + torch.randn(SG_SAMPLES, *xb.shape[1:], device=xb.device) * std).requires_grad_(True)
    return torch.autograd.grad(model(noisy)[:, target].sum(), noisy)[0].detach().mean(dim=0)

def raw_vanilla(model, x, target):
    xb = _xb(model, x).requires_grad_(True)
    return Saliency(model).attribute(xb, target=target, abs=False).detach().squeeze(0)

def raw_expgrad(model, x, target):
    xb    = _xb(model, x)
    bg    = torch.randn(EG_SAMPLES, *xb.shape[1:], device=xb.device)
    alpha = torch.rand(EG_SAMPLES, 1, 1, 1, device=xb.device)
    interp = (bg + alpha * (xb - bg)).requires_grad_(True)
    grads  = torch.autograd.grad(model(interp)[:, target].sum(), interp)[0]
    return (grads.detach() * (xb - bg)).mean(dim=0)

def raw_idg(model, x, target):
    xb = _xb(model, x).requires_grad_(True)
    g  = torch.autograd.grad(model(xb)[:, target].sum(), xb)[0]
    return (xb * g).detach().squeeze(0)

RAW_FN = {
    "IDG":          raw_idg,
    "ExpGrad":      raw_expgrad,
    "IG-zero":      raw_ig_zero,
    "Blur-IG":      raw_blur_ig,
    "SmoothGrad":   raw_smoothgrad,
    "Vanilla Grad": raw_vanilla,
}

# ── Cache helpers ──
def cache_load(name):
    p = CACHE_DIR / f"{name}.pkl"
    if p.exists() and not FORCE_RECOMPUTE:
        with open(p, "rb") as f:
            return pickle.load(f)
    return None

def cache_save(name, obj):
    with open(CACHE_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f)

def sem(arr):
    arr = np.asarray(arr, dtype=np.float64)
    return float(arr.std() / np.sqrt(max(len(arr), 1)))

def ci95(arr):
    arr = np.asarray(arr, dtype=np.float64)
    return float(1.96 * arr.std() / np.sqrt(max(len(arr), 1)))


def sensitivity_n(model, x, attr_map, target,
                  fractions=SENS_FRACTIONS, n_subsets=N_SENS_SUBSETS):
    H, W = attr_map.shape; n_pix = H * W; C = x.shape[1]
    attr_flat = attr_map.detach().cpu().numpy().ravel()
    with torch.no_grad():
        f_orig = model(x).softmax(-1)[0, target].item()
    x0 = x.view(C, n_pix)
    pccs = []
    for frac in fractions:
        n = max(1, int(frac * n_pix))
        subsets   = np.stack([np.random.choice(n_pix, n, replace=False)
                              for _ in range(n_subsets)])         # (S, n)
        attr_sums = attr_flat[subsets].sum(axis=1)               # (S,) vectorized
        # Vectorized zero-mask via scatter
        mask = torch.zeros(n_subsets, n_pix, dtype=torch.bool, device=x.device)
        rows = torch.arange(n_subsets, device=x.device).repeat_interleave(n)
        cols = torch.from_numpy(subsets.ravel()).to(x.device)
        mask[rows, cols] = True
        x_batch = x0.unsqueeze(0).expand(n_subsets, -1, -1).clone()
        x_batch[mask.unsqueeze(1).expand(-1, C, -1)] = 0.0
        with torch.no_grad():
            f_masked = model(x_batch.view(n_subsets, C, H, W)).softmax(-1)[:, target].cpu().numpy()
        r, _ = stats.pearsonr(attr_sums, f_orig - f_masked)
        pccs.append(r if not np.isnan(r) else 0.0)
    return np.array(pccs)

# ── Batched insertion / deletion ──
def insertion_deletion(model, x, attr_map, target, substrate,
                       n_steps=N_INSERTION_STEPS, batch_size=64):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]; n_pix = H * W
    order = attr_map.detach().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps); n_frames = n_steps + 1
    rank_step  = (torch.arange(n_pix) // pps + 1).clamp(max=n_steps)
    pixel_step = torch.zeros(n_pix, dtype=torch.long)
    pixel_step[order] = rank_step


    mask  = (pixel_step.view(1, -1) <= torch.arange(n_frames).view(-1, 1)
             ).float().unsqueeze(1).to(x.device)                 # (n_frames,1,n_pix)
    sub_f = substrate.view(1, C, n_pix)
    x_f   = x.view(1, C, n_pix)
    diff  = x_f - sub_f
    ins_batch = (sub_f + mask * diff).view(n_frames, C, H, W)
    del_batch = (x_f  - mask * diff).view(n_frames, C, H, W)

    with torch.no_grad():
        ins_sc, del_sc = [], []
        for i in range(0, n_frames, batch_size):
            ins_sc.append(model(ins_batch[i:i+batch_size]).softmax(-1)[:, target].cpu().numpy())
            del_sc.append(model(del_batch[i:i+batch_size]).softmax(-1)[:, target].cpu().numpy())
    ins = np.concatenate(ins_sc); dl = np.concatenate(del_sc)
    return float(np.trapz(ins, dx=1/n_steps)), float(np.trapz(dl, dx=1/n_steps)), ins, dl

def estimate_object_mask(x, attr_map):
    H, W = attr_map.shape
    a = np.abs(attr_map.detach().cpu().numpy())
    seed = (a >= np.percentile(a, 80)).astype(np.uint8)
    img_rgb = denormalize(x[0]).detach().cpu().permute(1, 2, 0).numpy()
    img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
    gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
    gc_mask[(a >= np.percentile(a, 95))] = cv2.GC_FGD
    edge_mask = np.zeros((H, W), dtype=np.uint8)
    border = max(H, W) // 10
    edge_mask[:border, :] = 1; edge_mask[-border:, :] = 1
    edge_mask[:, :border] = 1; edge_mask[:, -border:] = 1
    gc_mask[(edge_mask == 1) & (a < np.percentile(a, 10))] = cv2.GC_BGD
    try:
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_bgr, gc_mask, None, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)
        return np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
    except:
        return seed

def object_focus_ratio(attr_map, obj_mask):
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

def discriminative_concentration(attr_map, obj_mask):
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    if total < 1e-12: return 0.0
    in_obj   = a[obj_mask == 1].sum() / total
    obj_area = obj_mask.sum() / obj_mask.size
    return float(in_obj / max(obj_area, 1e-12))

def gini_coefficient(attr_map):
    a = np.abs(attr_map.detach().cpu().numpy().ravel())
    a = a[a > 0]
    if len(a) == 0: return 0.0
    a = np.sort(a); n = len(a)
    return float((2 * np.sum(np.arange(1, n + 1) * a)) / (n * np.sum(a)) - (n + 1) / n)

def sanity_similarity(map_a, map_b):
    a = map_a.detach().cpu().numpy().ravel()
    b = map_b.detach().cpu().numpy().ravel()
    if a.std() < 1e-9 or b.std() < 1e-9: return 0.0
    rho, _ = stats.spearmanr(a, b)
    return 0.0 if np.isnan(rho) else float(rho)
# ── Occlusion helpers ──
import math as _math

def ratio_to_patch(ratio, img_size=224):
    return max(2, int(round(_math.sqrt(ratio * img_size * img_size))))

def occlusion_map(model, x, target, ps, stride=OCCLUSION_STRIDE, batch_size=64):
    """Batched: ~batch_size patch positions per forward pass."""
    _, C, H, W = x.shape
    positions = [(h, w)
                 for h in range(0, H - ps + 1, stride)
                 for w in range(0, W - ps + 1, stride)]
    n_h = len(range(0, H - ps + 1, stride))
    n_w = len(range(0, W - ps + 1, stride))
    with torch.no_grad():
        f0 = model(x)[0, target].item()
        scores = []
        for i in range(0, len(positions), batch_size):
            bpos = positions[i : i + batch_size]
            batch = x.expand(len(bpos), -1, -1, -1).clone()
            for k, (h, w) in enumerate(bpos):
                batch[k, :, h : h + ps, w : w + ps] = 0.0
            scores.append(model(batch)[:, target].cpu().numpy())
    return (f0 - np.concatenate(scores)).reshape(n_h, n_w)

def attr_to_grid(attr_map, ps, stride, H, W):
    """Vectorized patch-mean via avg_pool2d."""
    a = attr_map.detach().abs().float().unsqueeze(0).unsqueeze(0)
    pooled = F.avg_pool2d(a, kernel_size=ps, stride=stride, padding=0)
    n_h = len(range(0, H - ps + 1, stride))
    n_w = len(range(0, W - ps + 1, stride))
    return pooled[0, 0, :n_h, :n_w].cpu().numpy()

# ── Infidelity ──
def infidelity(model, x, attr_map, target, n_perturbations=30, noise_sigma=0.1):
    """Batched: all perturbations in one forward pass."""
    with torch.no_grad():
        f_x  = model(x)[0, target].item()
        I    = torch.randn(n_perturbations, *x.shape[1:], device=x.device) * noise_sigma
        f_xp = model(x - I)[:, target].cpu().numpy()                        # (N,)
        dot  = (I * attr_map.unsqueeze(0).unsqueeze(0)).view(n_perturbations, -1).sum(dim=1).cpu().numpy()
    return float(np.mean((dot - (f_x - f_xp)) ** 2))

print("Metric functions ready (fully batched)")


#METRICS

#1. Sensitivity_n

In [ ]:
# ── Metric 1: Sensitivity-n ──
import pandas as pd

_cache = CACHE_DIR / "sens_n.pkl"
_csv   = CACHE_DIR / "sens_n.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_sens, all_sens_mean = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_sens = pd.read_csv(_csv)
    all_sens      = {m: [] for m in methods_all}
    all_sens_mean = {m: [] for m in methods_all}
    for m in methods_all:
        sub = df_sens[df_sens.method == m]
        for _, grp in sub.groupby("img_i"):
            curve = grp.sort_values("frac")["pcc"].to_numpy()
            all_sens[m].append(curve)
            all_sens_mean[m].append(float(curve.mean()))
    print(f"[csv] Reconstructed from {_csv}")

else:
    all_sens      = {m: [] for m in methods_all}
    all_sens_mean = {m: [] for m in methods_all}

    for idx in tqdm(eval_idx, desc="sensitivity-n"):
        r = results[idx]; x, tgt = r["x"], r["target"]

        for sigma_name in ["adaptive", "σ=0.25"]:
            sf = get_sigma_final(sigma_name, model, x, tgt)
            _, amap = compute_klig(model, x, tgt, sf)
            curve = sensitivity_n(model, x, amap, tgt)
            m = f"KL-IG ({sigma_name})"
            all_sens[m].append(curve)
            all_sens_mean[m].append(float(curve.mean()))

        for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad", "Blur-IG"]:
            amap  = COMPUTE_FN[m](model, x, tgt)
            curve = sensitivity_n(model, x, amap, tgt)
            all_sens[m].append(curve)
            all_sens_mean[m].append(float(curve.mean()))

    # save pickle
    with open(_cache, "wb") as f:
        pickle.dump((all_sens, all_sens_mean), f)

    # save CSV (long format — survives runtime reset)
    rows = []
    for m in methods_all:
        for img_i, curve in enumerate(all_sens[m]):
            for frac, pcc in zip(SENS_FRACTIONS, curve):
                rows.append({"method": m, "img_i": img_i, "frac": frac, "pcc": float(pcc)})
    pd.DataFrame(rows).to_csv(_csv, index=False)

    # save summary CSV
    pd.DataFrame([{
        "method":   m,
        "mean_pcc": round(np.mean(all_sens_mean[m]), 4),
        "ci95":     round(ci95(all_sens_mean[m]), 4),
        "std":      round(np.std(all_sens_mean[m]), 4),
        "n_images": len(all_sens_mean[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "sens_n_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")

# ── Plot: line with 95% CI band ──
n = len(next(iter(all_sens.values())))
fracs = np.array(SENS_FRACTIONS)
fig, ax = plt.subplots(figsize=(13, 6), facecolor="white")
for m in methods_all:
    stacked = np.stack(all_sens[m])
    mu = stacked.mean(axis=0)
    ci = 1.96 * stacked.std(axis=0) / np.sqrt(n)
    ax.plot(fracs, mu, "o-", color=COLORS_ALL[m], lw=2, ms=4,
            label=f"{m} ({mu.mean():.3f})")
    ax.fill_between(fracs, mu - ci, mu + ci, color=COLORS_ALL[m], alpha=0.15)
ax.axhline(0, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("Fraction of pixels perturbed", fontsize=11)
ax.set_ylabel("PCC", fontsize=11)
ax.set_title(f"Sensitivity-n (↑) ± 95% CI   n={n}", fontweight="bold", fontsize=12)
ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Sensitivity-n: per-image mean PCC distribution ──
import pandas as pd

_csv = CACHE_DIR / "sens_n.csv"
df_sens = pd.read_csv(_csv)
# per-image mean PCC = average PCC across fractions for each (method, img_i)
per_img = (df_sens.groupby(["method", "img_i"])["pcc"]
                  .mean()
                  .reset_index()
                  .rename(columns={"pcc": "mean_pcc"}))

fig, ax = plt.subplots(figsize=(11, 5), facecolor="white")
data = [per_img[per_img.method == m]["mean_pcc"].to_numpy() for m in methods_all]
bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color="black", lw=2))
for patch, m in zip(bp["boxes"], methods_all):
    patch.set_facecolor(COLORS_ALL[m]); patch.set_alpha(0.75)
ax.set_xticks(range(1, len(methods_all)+1))
ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("Mean PCC")
ax.set_title(f"Sensitivity-n distribution   n={per_img['img_i'].nunique()}", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n{'Method':<25} {'Mean PCC':>10} {'95% CI':>10} {'n':>6}")
print("-" * 54)
rows_table = []
for m in methods_all:
    vals = per_img[per_img.method == m]["mean_pcc"].to_numpy()
    rows_table.append({"Method": m, "Mean PCC": round(vals.mean(), 4),
                       "95% CI": round(ci95(vals), 4), "n": len(vals)})
    print(f"{m:<25} {vals.mean():>10.4f} {ci95(vals):>10.4f} {len(vals):>6}")

display(pd.DataFrame(rows_table).set_index("Method"))


#2.Insertion Deletion


In [ ]:
def insertion_deletion(model, x, attr_map, target,
                       n_steps=N_INSERTION_STEPS, batch_size=64):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]; n_pix = H * W
    order = attr_map.detach().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps); n_frames = n_steps + 1

    rank_step  = (torch.arange(n_pix) // pps + 1).clamp(max=n_steps)
    pixel_step = torch.zeros(n_pix, dtype=torch.long)
    pixel_step[order] = rank_step

    mask = (pixel_step.view(1, -1) <= torch.arange(n_frames).view(-1, 1)
            ).float().unsqueeze(1).to(x.device)   # (n_frames,1,n_pix)

    x_f      = x.view(1, C, n_pix)
    blur_sub = make_blur_baseline(x).view(1, C, n_pix)   # insertion: blur → image
    zero_sub = torch.zeros_like(x_f)                      # deletion:  image → zeros

    ins_batch = (blur_sub + mask * (x_f - blur_sub)).view(n_frames, C, H, W)
    del_batch = (x_f      - mask * (x_f - zero_sub)).view(n_frames, C, H, W)

    with torch.no_grad():
        ins_sc, del_sc = [], []
        for i in range(0, n_frames, batch_size):
            ins_sc.append(model(ins_batch[i:i+batch_size]).softmax(-1)[:, target].cpu().numpy())
            del_sc.append(model(del_batch[i:i+batch_size]).softmax(-1)[:, target].cpu().numpy())
    ins = np.concatenate(ins_sc); dl = np.concatenate(del_sc)
    return float(np.trapz(ins, dx=1/n_steps)), float(np.trapz(dl, dx=1/n_steps)), ins, dl

In [ ]:
# ── Metric 2: Insertion / Deletion (blur substrate for insertion, zeros for deletion) ──
import pandas as pd

_cache = CACHE_DIR / "ins_del_blur.pkl"
_csv   = CACHE_DIR / "ins_del_blur.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        ins_auc, del_auc, ins_curves, del_curves = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_id = pd.read_csv(_csv)
    ins_auc    = {m: [] for m in methods_all}
    del_auc    = {m: [] for m in methods_all}
    ins_curves = {m: [] for m in methods_all}
    del_curves = {m: [] for m in methods_all}
    for m in methods_all:
        sub = df_id[df_id.method == m]
        for _, grp in sub.groupby("img_i"):
            grp_s = grp.sort_values("step")
            ins_curves[m].append(grp_s["ins"].to_numpy())
            del_curves[m].append(grp_s["del"].to_numpy())
            ins_auc[m].append(float(np.trapz(grp_s["ins"].to_numpy(), dx=1/N_INSERTION_STEPS)))
            del_auc[m].append(float(np.trapz(grp_s["del"].to_numpy(), dx=1/N_INSERTION_STEPS)))
    print(f"[csv] Reconstructed from {_csv}")

else:
    ins_auc    = {m: [] for m in methods_all}
    del_auc    = {m: [] for m in methods_all}
    ins_curves = {m: [] for m in methods_all}
    del_curves = {m: [] for m in methods_all}

    for idx in tqdm(eval_idx, desc="insertion/deletion"):
        r = results[idx]; x, tgt = r["x"], r["target"]

        for sigma_name in ["adaptive", "σ=0.25"]:
            sf  = get_sigma_final(sigma_name, model, x, tgt)
            raw = raw_klig(model, x, tgt, sf)
            m   = f"KL-IG ({sigma_name})"
            ia, da, ic, dc = insertion_deletion(model, x, sum_collapse(raw), tgt)
            ins_auc[m].append(ia); del_auc[m].append(da)
            ins_curves[m].append(ic); del_curves[m].append(dc)

        for m, fn in RAW_FN.items():
            raw = fn(model, x, tgt)
            ia, da, ic, dc = insertion_deletion(model, x, sum_collapse(raw), tgt)
            ins_auc[m].append(ia); del_auc[m].append(da)
            ins_curves[m].append(ic); del_curves[m].append(dc)

    with open(_cache, "wb") as f:
        pickle.dump((ins_auc, del_auc, ins_curves, del_curves), f)

    rows = []
    for m in methods_all:
        for img_i, (ic, dc) in enumerate(zip(ins_curves[m], del_curves[m])):
            for step, (i_s, d_s) in enumerate(zip(ic, dc)):
                rows.append({"method": m, "img_i": img_i, "step": step,
                             "ins": float(i_s), "del": float(d_s)})
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":   m,
        "ins_mean": round(np.mean(ins_auc[m]), 4),
        "ins_ci95": round(ci95(ins_auc[m]), 4),
        "del_mean": round(np.mean(del_auc[m]), 4),
        "del_ci95": round(ci95(del_auc[m]), 4),
        "n_images": len(ins_auc[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "ins_del_summary_blur.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")

In [ ]:
# ── I/D AUC bars + curves (reads from CSV) ──
import pandas as pd

df_id = pd.read_csv(CACHE_DIR / "ins_del_blur.csv")

# ── Compute AUC per image from curves ──
records = []
for m in methods_all:
    sub = df_id[df_id.method == m]
    ins_aucs, del_aucs = [], []
    for _, grp in sub.groupby("img_i"):
        grp_s = grp.sort_values("step")
        ins_aucs.append(np.trapz(grp_s["ins"].to_numpy(), dx=1/N_INSERTION_STEPS))
        del_aucs.append(np.trapz(grp_s["del"].to_numpy(), dx=1/N_INSERTION_STEPS))
    records.append({
        "method":   m,
        "ins_mean": np.mean(ins_aucs),
        "ins_ci95": 1.96 * np.std(ins_aucs) / np.sqrt(len(ins_aucs)),
        "del_mean": np.mean(del_aucs),
        "del_ci95": 1.96 * np.std(del_aucs) / np.sqrt(len(del_aucs)),
        "n_images": len(ins_aucs),
    })

df_sum = pd.DataFrame(records).set_index("method").reindex(methods_all)
n = int(df_sum["n_images"].iloc[0])

ins_means = df_sum["ins_mean"].to_numpy()
del_means = df_sum["del_mean"].to_numpy()
ins_cis   = df_sum["ins_ci95"].to_numpy()
del_cis   = df_sum["del_ci95"].to_numpy()
x_pos     = np.arange(len(methods_all))

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(13, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    c = COLORS_ALL[m]
    ax.bar(xi - 0.2, ins_means[xi], 0.4, color=c, alpha=0.85,
           yerr=ins_cis[xi], capsize=3, error_kw=dict(lw=1))
    ax.bar(xi + 0.2, del_means[xi], 0.4, color=c, alpha=0.45, hatch="//",
           yerr=del_cis[xi], capsize=3, error_kw=dict(lw=1))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="gray", alpha=0.85,             label="Insertion ↑"),
                   Patch(facecolor="gray", alpha=0.45, hatch="//", label="Deletion ↓")])
ax.set_xticks(x_pos); ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("AUC")
ax.set_title(f"Insertion / Deletion  n={n}", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

# ── Mean curves ──
ins_curves_p = {m: [] for m in methods_all}
del_curves_p = {m: [] for m in methods_all}
for m in methods_all:
    sub = df_id[df_id.method == m]
    for _, grp in sub.groupby("img_i"):
        grp_s = grp.sort_values("step")
        ins_curves_p[m].append(grp_s["ins"].to_numpy())
        del_curves_p[m].append(grp_s["del"].to_numpy())

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor="white")
steps = np.linspace(0, 1, N_INSERTION_STEPS + 1)
for m in methods_all:
    ic = np.stack(ins_curves_p[m]); dc = np.stack(del_curves_p[m])
    mu_i = ic.mean(0); ci_i = 1.96 * ic.std(0) / np.sqrt(n)
    mu_d = dc.mean(0); ci_d = 1.96 * dc.std(0) / np.sqrt(n)
    axes[0].plot(steps, mu_i, color=COLORS_ALL[m], lw=1.5, label=m)
    axes[0].fill_between(steps, mu_i - ci_i, mu_i + ci_i,
                         color=COLORS_ALL[m], alpha=0.1)
    axes[1].plot(steps, mu_d, color=COLORS_ALL[m], lw=1.5, label=m)
    axes[1].fill_between(steps, mu_d - ci_d, mu_d + ci_d,
                         color=COLORS_ALL[m], alpha=0.1)
for axi, title in zip(axes, ["Insertion ↑", "Deletion ↓"]):
    axi.set_xlabel("Fraction of pixels revealed")
    axi.set_ylabel("P(target)")
    axi.set_title(title, fontweight="bold")
    axi.legend(fontsize=7)
    axi.grid(alpha=0.3)
plt.suptitle(f"I/D curves   n={n}", fontweight="bold")
plt.tight_layout(); plt.show()

# ── Summary table ──
rows_t = [{"Method":  m,
           "Ins AUC": round(float(df_sum.loc[m, "ins_mean"]), 4),
           "± CI":    round(float(df_sum.loc[m, "ins_ci95"]), 4),
           "Del AUC": round(float(df_sum.loc[m, "del_mean"]), 4),
           "± CI ":   round(float(df_sum.loc[m, "del_ci95"]), 4),
           "Ins−Del": round(float(df_sum.loc[m, "ins_mean"]) - float(df_sum.loc[m, "del_mean"]), 4),
          } for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style
    .format("{:.4f}")
    .background_gradient(subset=["Ins AUC"],  cmap="Greens")
    .background_gradient(subset=["Del AUC"],  cmap="Reds_r")
    .background_gradient(subset=["Ins−Del"],  cmap="PuBu")
    .set_caption(f"Insertion / Deletion AUC   n={n}"))

# 3. OFR

In [ ]:
# ── OFR Qualitative: pos/neg heatmap helper (cividis) ──
def plot_attr_pn(ax_pos, ax_neg, x, raw_attr, obj_mask=None):
    img  = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    gray = np.mean(img, axis=2)
    for ax, a_hw, cmap in [
        (ax_pos, raw_attr.clamp(min=0).sum(dim=0),       "cividis"),
        (ax_neg, raw_attr.clamp(max=0).abs().sum(dim=0), "cividis_r"),
    ]:
        a = a_hw.detach().cpu().numpy()
        a = gaussian_filter(a, sigma=2)
        vmax = np.percentile(a, 99); vmax = vmax if vmax > 1e-12 else 1.0
        ax.imshow(gray, cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.clip(a / vmax, 0, 1), cmap=cmap, vmin=0, vmax=1, alpha=0.65)
        if obj_mask is not None:
            cnts, _ = cv2.findContours(obj_mask.astype(np.uint8),
                                       cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts:
                pts = cnt[:, 0, :]
                ax.plot(np.append(pts[:,0], pts[0,0]),
                        np.append(pts[:,1], pts[0,1]),
                        color="white", lw=1.2)
        ax.set_xticks([]); ax.set_yticks([])

In [ ]:
# ── Score top-N images by KL-IG OFR margin ──
from scipy.ndimage import gaussian_filter
SCORE_POOL = eval_idx[:min(60, len(eval_idx))]
scored = []

for idx in tqdm(SCORE_POOL, desc="scoring images for OFR figure"):
    r = results[idx]; x, tgt = r["x"], r["target"]
    sf = get_sigma_final("adaptive", model, x, tgt)
    raw_k = raw_klig(model, x, tgt, sf)
    klig_amap = sum_collapse(raw_k)
    obj_mask = estimate_object_mask(x, klig_amap)
    frac = obj_mask.mean()
    if frac < 0.02 or frac > 0.85: continue
    ofr_klig = object_focus_ratio(klig_amap, obj_mask)
    ofr_base = [object_focus_ratio(sum_collapse(RAW_FN[m](model, x, tgt)), obj_mask)
                for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad", "Blur-IG"]]
    margin = ofr_klig - float(np.mean(ofr_base))
    score  = ofr_klig + 0.5 * margin
    scored.append((score, ofr_klig, margin, idx, raw_k, obj_mask))

scored.sort(key=lambda t: -t[0])
print("Top 5:")
for s, ok, mg, idx, _, _ in scored[:5]:
    print(f"  idx={idx:4d}  KL-IG OFR={ok:.3f}  margin={mg:+.3f}  label={results[idx]['label_str']}")

In [ ]:
# ── OFR Qualitative: top-5 images, pos/neg side-by-side, cividis ──
n_examples = 5
top = scored[:n_examples]

VIS_METHODS = ["KL-IG (adaptive)", "KL-IG (σ=0.25)",
               "IDG", "ExpGrad", "IG-zero", "Blur-IG", "SmoothGrad", "Vanilla Grad"]
N_COLS = 1 + len(VIS_METHODS)
fig, axes = plt.subplots(n_examples * 2, N_COLS,
                         figsize=(N_COLS * 2.2, n_examples * 4.0), facecolor="white")
if n_examples == 1: axes = axes[np.newaxis, :]

for row, (_, ofr_k, _, idx, raw_k, obj_mask) in enumerate(top):
    r = results[idx]; x, tgt = r["x"], r["target"]
    img_np = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    r_pos, r_neg = row * 2, row * 2 + 1
    for ri in [r_pos, r_neg]:
        axes[ri, 0].imshow(img_np); axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    axes[r_pos, 0].set_ylabel(f"{r['label_str'][:16]}\n(+)", fontsize=7,
                               rotation=0, labelpad=70, va="center", fontweight="bold")
    axes[r_neg, 0].set_ylabel("(−)", fontsize=9, rotation=0, labelpad=70, va="center")

    sf025 = get_sigma_final("σ=0.25", model, x, tgt)
    raw_maps = {"KL-IG (adaptive)": raw_k,
                "KL-IG (σ=0.25)":   raw_klig(model, x, tgt, sf025)}
    for m, fn in RAW_FN.items():
        raw_maps[m] = fn(model, x, tgt)

    for col, m in enumerate(VIS_METHODS, start=1):
        plot_attr_pn(axes[r_pos, col], axes[r_neg, col], x, raw_maps[m], obj_mask)
        if row == 0:
            axes[r_pos, col].set_title(m, fontsize=8, fontweight="bold")

axes[0, 0].set_title("Original", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("ofr_top5_signed.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Metric 5: Object Focus Ratio ──
import pandas as pd

_cache = CACHE_DIR / "ofr.pkl"
_csv   = CACHE_DIR / "ofr.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_ofr = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_ofr = pd.read_csv(_csv)
    all_ofr = {m: df_ofr[df_ofr.method == m].sort_values("img_i")["ofr"].to_list()
               for m in methods_all}
    print(f"[csv] Reconstructed from {_csv}")

else:
    all_ofr = {m: [] for m in methods_all}
    for img_i, idx in enumerate(tqdm(eval_idx, desc="OFR")):
        r = results[idx]; x, tgt = r["x"], r["target"]
        sf0 = get_sigma_final("adaptive", model, x, tgt)
        ref_raw = raw_klig(model, x, tgt, sf0)
        obj_mask = estimate_object_mask(x, sum_collapse(ref_raw))

        for sigma_name in ["adaptive", "σ=0.25"]:
            sf  = get_sigma_final(sigma_name, model, x, tgt)
            raw = raw_klig(model, x, tgt, sf)
            all_ofr[f"KL-IG ({sigma_name})"].append(object_focus_ratio(sum_collapse(raw), obj_mask))
        for m, fn in RAW_FN.items():
            raw = fn(model, x, tgt)
            all_ofr[m].append(object_focus_ratio(sum_collapse(raw), obj_mask))

    with open(_cache, "wb") as f:
        pickle.dump(all_ofr, f)

    rows = [{"method": m, "img_i": i, "ofr": float(v)}
            for m in methods_all for i, v in enumerate(all_ofr[m])]
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":   m,
        "mean_ofr": round(np.mean(all_ofr[m]), 4),
        "ci95":     round(ci95(all_ofr[m]), 4),
        "n_images": len(all_ofr[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "ofr_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")


In [ ]:
# ── OFR: plot from CSV ──
import pandas as pd

df_sum = pd.read_csv(CACHE_DIR / "ofr_summary.csv").set_index("method").reindex(methods_all)
n = int(df_sum["n_images"].iloc[0])
means = df_sum["mean_ofr"].to_numpy()

fig, ax = plt.subplots(figsize=(13, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    ax.bar(xi, means[xi], color=COLORS_ALL[m], edgecolor="black", alpha=0.85)
    ax.text(xi, means[xi] + 0.012, f"{means[xi]:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(methods_all))); ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("OFR"); ax.set_ylim(0, 1.1)
ax.set_title(f"Object Focus Ratio (\u2191, sum-collapse)   n={n}", fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

rows_t = [{"Method": m,
           "Mean OFR": round(float(df_sum.loc[m, "mean_ofr"]), 4),
           "\u00b1 95% CI": round(float(df_sum.loc[m, "ci95"]), 4)} for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style.format("{:.4f}").background_gradient(cmap="YlGn")
        .set_caption(f"Object Focus Ratio    n={n}"))

In [ ]:
# ── Metric 6: Discriminative Concentration (sum-collapse) ──
import pandas as pd

_cache = CACHE_DIR / "dc.pkl"
_csv   = CACHE_DIR / "dc.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_dc = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_dc = pd.read_csv(_csv)
    all_dc = {m: df_dc[df_dc.method == m].sort_values("img_i")["dc"].to_list()
              for m in methods_all}
    print(f"[csv] Reconstructed from {_csv}")

else:
    all_dc = {m: [] for m in methods_all}
    for img_i, idx in enumerate(tqdm(eval_idx, desc="disc-conc")):
        r = results[idx]; x, tgt = r["x"], r["target"]
        sf0 = get_sigma_final("adaptive", model, x, tgt)
        ref_raw = raw_klig(model, x, tgt, sf0)
        obj_mask = estimate_object_mask(x, sum_collapse(ref_raw))

        for sigma_name in ["adaptive", "σ=0.25"]:
            sf  = get_sigma_final(sigma_name, model, x, tgt)
            raw = raw_klig(model, x, tgt, sf)
            all_dc[f"KL-IG ({sigma_name})"].append(
                discriminative_concentration(sum_collapse(raw), obj_mask))
        for m, fn in RAW_FN.items():
            raw = fn(model, x, tgt)
            all_dc[m].append(discriminative_concentration(sum_collapse(raw), obj_mask))

    with open(_cache, "wb") as f:
        pickle.dump(all_dc, f)

    rows = [{"method": m, "img_i": i, "dc": float(v)}
            for m in methods_all for i, v in enumerate(all_dc[m])]
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":   m,
        "mean_dc":  round(np.mean(all_dc[m]), 4),
        "ci95":     round(ci95(all_dc[m]), 4),
        "n_images": len(all_dc[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "dc_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")


In [ ]:
# ── DC: plot from CSV ──
import pandas as pd

df_sum = pd.read_csv(CACHE_DIR / "dc_summary.csv").set_index("method").reindex(methods_all)
n = int(df_sum["n_images"].iloc[0])
means = df_sum["mean_dc"].to_numpy()

fig, ax = plt.subplots(figsize=(13, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    ax.bar(xi, means[xi], color=COLORS_ALL[m], edgecolor="black", alpha=0.85)
    ax.text(xi, means[xi] + 0.03, f"{means[xi]:.2f}", ha="center", fontsize=9)
ax.set_xticks(range(len(methods_all))); ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("DC")
ax.axhline(1.0, color="gray", ls="--", alpha=0.5, label="uniform baseline")
ax.set_title(f"Discriminative Concentration (\u2191, sum-collapse)   n={n}", fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

rows_t = [{"Method": m,
           "Mean DC": round(float(df_sum.loc[m, "mean_dc"]), 4),
           "\u00b1 95% CI": round(float(df_sum.loc[m, "ci95"]), 4)} for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style.format("{:.4f}").background_gradient(cmap="Oranges")
        .set_caption(f"Discriminative Concentration (\u2191, sum-collapse)   n={n}"))

#4.Occlusion Corr

In [ ]:
# ── Metric 3: Occlusion Correlation vs Ratio ──
import math, pandas as pd

_cache = CACHE_DIR / "occlusion.pkl"
_csv   = CACHE_DIR / "occlusion.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        occ_by_ratio = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_occ = pd.read_csv(_csv)
    occ_by_ratio = {m: [] for m in methods_all}
    for m in methods_all:
        sub = df_occ[df_occ.method == m]
        for ratio in OCCLUSION_RATIOS:
            r_grp = sub[sub.ratio == ratio].sort_values("img_i")
            occ_by_ratio[m].append(r_grp["rho"].to_list())
    print(f"[csv] Reconstructed from {_csv}")

else:
    attr_cache = {}
    for idx in tqdm(eval_idx, desc="attrs (once)"):
        r = results[idx]; x, tgt = r["x"], r["target"]
        cache = {}
        for sigma_name in ["adaptive", "\u03c3=0.25"]:
            sf = get_sigma_final(sigma_name, model, x, tgt)
            _, amap = compute_klig(model, x, tgt, sf)
            cache[f"KL-IG ({sigma_name})"] = amap
        for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad", "Blur-IG"]:
            cache[m] = COMPUTE_FN[m](model, x, tgt)
        attr_cache[idx] = cache

    occ_by_ratio = {m: [[] for _ in OCCLUSION_RATIOS] for m in methods_all}
    for ri, ratio in enumerate(tqdm(OCCLUSION_RATIOS, desc="occlusion sweep")):
        ps = ratio_to_patch(ratio)
        for idx in eval_idx:
            r = results[idx]; x, tgt = r["x"], r["target"]
            _, _, H, W = x.shape
            om = occlusion_map(model, x, tgt, ps=ps).ravel()
            for m in methods_all:
                ag = attr_to_grid(attr_cache[idx][m], ps, OCCLUSION_STRIDE, H, W).ravel()
                rho, _ = stats.spearmanr(om, ag)
                occ_by_ratio[m][ri].append(0.0 if np.isnan(rho) else rho)

    with open(_cache, "wb") as f:
        pickle.dump(occ_by_ratio, f)

    rows = []
    for m in methods_all:
        for ri, ratio in enumerate(OCCLUSION_RATIOS):
            for img_i, rho in enumerate(occ_by_ratio[m][ri]):
                rows.append({"method": m, "img_i": img_i, "ratio": ratio, "rho": float(rho)})
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":   m,
        "mean_rho": round(np.mean([np.mean(occ_by_ratio[m][ri]) for ri in range(len(OCCLUSION_RATIOS))]), 4),
        "ci95":     round(ci95(np.array(occ_by_ratio[m]).mean(axis=0)), 4),
        "n_images": len(occ_by_ratio[m][0]),
    } for m in methods_all]).to_csv(CACHE_DIR / "occlusion_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")

# ── Plot per-ratio with 95% CI bands ──
n = len(occ_by_ratio[methods_all[0]][0])
fig, ax = plt.subplots(figsize=(13, 6), facecolor="white")
for m in methods_all:
    arr = np.array(occ_by_ratio[m])  # (n_ratios, n_imgs)
    mu  = arr.mean(axis=1)
    ci  = 1.96 * arr.std(axis=1) / np.sqrt(n)
    ax.plot(OCCLUSION_RATIOS, mu, "o-", color=COLORS_ALL[m], lw=2, ms=5,
            label=f"{m} ({mu.mean():.3f})")
    ax.fill_between(OCCLUSION_RATIOS, mu - ci, mu + ci, color=COLORS_ALL[m], alpha=0.15)
ax.set_xlabel("Occlusion ratio"); ax.set_ylabel("Spearman \u03c1")
ax.set_title(f"Occlusion Correlation (\u2191) \u00b1 95% CI   n={n}", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Occlusion Correlation: all plots + table (loads from CSV) ──
import pandas as pd

df_sum = pd.read_csv(CACHE_DIR / "occlusion_summary.csv").set_index("method").reindex(methods_all)
df_occ = pd.read_csv(CACHE_DIR / "occlusion.csv")
n      = int(df_sum["n_images"].iloc[0])

# Reconstruct per-ratio arrays
occ_by_ratio_p = {m: [] for m in methods_all}
for m in methods_all:
    sub = df_occ[df_occ.method == m]
    for ratio in OCCLUSION_RATIOS:
        occ_by_ratio_p[m].append(sub[sub.ratio == ratio].sort_values("img_i")["rho"].to_numpy())

# ── (1) Avg bar chart ──
means = df_sum["mean_rho"].to_numpy()
cis   = df_sum["ci95"].to_numpy()
fig, ax = plt.subplots(figsize=(12, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    ax.bar(xi, means[xi],
           color=COLORS_ALL[m], edgecolor="black", alpha=0.9)
    ax.text(xi, means[xi] + 0.003,
            f"{means[xi]:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(methods_all))); ax.set_xticklabels(methods_all, rotation=25, ha="right")
ax.set_ylabel("Avg Spearman \u03c1")
ax.set_title(f"Occlusion Correlation Avg   n={n}", fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

# ── (2) Per-ratio line with 95% CI band ──
fig, ax = plt.subplots(figsize=(13, 6), facecolor="white")
for m in methods_all:
    arr = np.array(occ_by_ratio_p[m])
    mu  = arr.mean(axis=1); ci = 1.96 * arr.std(axis=1) / np.sqrt(n)
    ax.plot(OCCLUSION_RATIOS, mu, "o-", color=COLORS_ALL[m], lw=2, ms=5,
            label=f"{m} ({mu.mean():.3f})")
ax.set_xlabel("Occlusion ratio"); ax.set_ylabel("Spearman \u03c1")
ax.set_title(f"Occlusion Correlation (\u2191)   n={n}", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ── (3) Per-ratio grouped bar chart ──
n_methods = len(methods_all); n_ratios = len(OCCLUSION_RATIOS)
bar_w = 0.8 / n_methods; x_centers = np.arange(n_ratios)
fig, ax = plt.subplots(figsize=(15, 6), facecolor="white")
for k, m in enumerate(methods_all):
    arr = np.array(occ_by_ratio_p[m])
    mu  = arr.mean(axis=1); ci = 1.96 * arr.std(axis=1) / np.sqrt(n)
    offsets = x_centers + (k - n_methods / 2) * bar_w + bar_w / 2
    for xi, (off, mu_i, ci_i) in enumerate(zip(offsets, mu, ci)):
        ax.bar(off, mu_i, bar_w,
               color=COLORS_ALL[m], alpha=0.9,
               label=m if xi == 0 else None)
ax.set_xticks(x_centers); ax.set_xticklabels([f"{int(r*100)}%" for r in OCCLUSION_RATIOS])
ax.set_xlabel("Occlusion ratio"); ax.set_ylabel("Spearman \u03c1")
ax.set_title(f"Occlusion Correlation per Ratio   n={n}", fontweight="bold")
ax.legend(fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

# ── Summary table ──
rows_t = [{"Method": m,
           "Avg \u03c1":    round(float(df_sum.loc[m, "mean_rho"]), 4),
           "\u00b1 95% CI": round(float(df_sum.loc[m, "ci95"]), 4),
          } for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style
    .format("{:.4f}")
    .background_gradient(cmap="YlGn")
    .set_caption(f"Occlusion Correlation (\u2191)   n={n}"))


#5. Sparsity

In [ ]:
# ── Metric 7: Gini Coefficient (absmax-collapse — sparsity metric) ──
import pandas as pd

_cache = CACHE_DIR / "gini.pkl"
_csv   = CACHE_DIR / "gini.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_gini = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_g = pd.read_csv(_csv)
    all_gini = {m: df_g[df_g.method == m].sort_values("img_i")["gini"].to_list()
                for m in methods_all}
    print(f"[csv] Reconstructed from {_csv}")

else:
    all_gini = {m: [] for m in methods_all}
    for img_i, idx in enumerate(tqdm(eval_idx, desc="gini")):
        r = results[idx]; x, tgt = r["x"], r["target"]
        for sigma_name in ["adaptive", "σ=0.25"]:
            sf  = get_sigma_final(sigma_name, model, x, tgt)
            raw = raw_klig(model, x, tgt, sf)
            all_gini[f"KL-IG ({sigma_name})"].append(gini_coefficient(absmax_collapse(raw)))
        for m, fn in RAW_FN.items():
            raw = fn(model, x, tgt)
            all_gini[m].append(gini_coefficient(absmax_collapse(raw)))

    with open(_cache, "wb") as f:
        pickle.dump(all_gini, f)

    rows = [{"method": m, "img_i": i, "gini": float(v)}
            for m in methods_all for i, v in enumerate(all_gini[m])]
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":    m,
        "mean_gini": round(np.mean(all_gini[m]), 4),
        "ci95":      round(ci95(all_gini[m]), 4),
        "n_images":  len(all_gini[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "gini_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")


In [ ]:
# ── Gini: plot from CSV ──
import pandas as pd

df_sum = pd.read_csv(CACHE_DIR / "gini_summary.csv").set_index("method").reindex(methods_all)
n = int(df_sum["n_images"].iloc[0])
means = df_sum["mean_gini"].to_numpy()

fig, ax = plt.subplots(figsize=(13, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    ax.bar(xi, means[xi], color=COLORS_ALL[m], edgecolor="black", alpha=0.85)
    ax.text(xi, means[xi] + 0.01, f"{means[xi]:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(methods_all))); ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("Gini Coefficient")
ax.set_title(f"Attribution Sparsity — absmax-collapse   n={n}", fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

rows_t = [{"Method": m,
           "Gini": round(float(df_sum.loc[m, "mean_gini"]), 4),
           "\u00b1 95% CI": round(float(df_sum.loc[m, "ci95"]), 4)} for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style.format("{:.4f}").background_gradient(cmap="Purples")
        .set_caption(f"Attribution Sparsity (\u2191 = more sparse) — absmax-collapse   n={n}"))

#6. Infidelity

In [ ]:
# ── Metric 8: Infidelity (sum-collapse) ──
import pandas as pd

_cache = CACHE_DIR / "infid.pkl"
_csv   = CACHE_DIR / "infid.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_infid = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_inf = pd.read_csv(_csv)
    all_infid = {m: df_inf[df_inf.method == m].sort_values("img_i")["infid"].to_list()
                 for m in methods_all}
    print(f"[csv] Reconstructed from {_csv}")

else:
    all_infid = {m: [] for m in methods_all}
    for img_i, idx in enumerate(tqdm(eval_idx, desc="infidelity")):
        r = results[idx]; x, tgt = r["x"], r["target"]
        for sigma_name in ["adaptive", "σ=0.25"]:
            sf  = get_sigma_final(sigma_name, model, x, tgt)
            raw = raw_klig(model, x, tgt, sf)
            all_infid[f"KL-IG ({sigma_name})"].append(
                infidelity(model, x, sum_collapse(raw), tgt))
        for m, fn in RAW_FN.items():
            raw = fn(model, x, tgt)
            all_infid[m].append(infidelity(model, x, sum_collapse(raw), tgt))

    with open(_cache, "wb") as f:
        pickle.dump(all_infid, f)

    rows = [{"method": m, "img_i": i, "infid": float(v)}
            for m in methods_all for i, v in enumerate(all_infid[m])]
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method":     m,
        "mean_infid": round(np.mean(all_infid[m]), 5),
        "ci95":       round(ci95(all_infid[m]), 5),
        "n_images":   len(all_infid[m]),
    } for m in methods_all]).to_csv(CACHE_DIR / "infid_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")


In [ ]:
# ── Infidelity: plot from CSV ──
import pandas as pd

df_sum = pd.read_csv(CACHE_DIR / "infid_summary.csv").set_index("method").reindex(methods_all)
n = int(df_sum["n_images"].iloc[0])
means = df_sum["mean_infid"].to_numpy()

fig, ax = plt.subplots(figsize=(13, 5), facecolor="white")
for xi, m in enumerate(methods_all):
    ax.bar(xi, means[xi], color=COLORS_ALL[m], edgecolor="black", alpha=0.85)
    ax.text(xi, means[xi] * 1.02, f"{means[xi]:.4f}", ha="center", fontsize=8)
ax.set_xticks(range(len(methods_all))); ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("Infidelity \u2193")
ax.set_title(f"Infidelity (\u2193, sum-collapse)   n={n}", fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

rows_t = [{"Method": m,
           "Infidelity": round(float(df_sum.loc[m, "mean_infid"]), 5),
           "\u00b1 95% CI": round(float(df_sum.loc[m, "ci95"]), 5)} for m in methods_all]
df_t = pd.DataFrame(rows_t).set_index("Method")
display(df_t.style.format("{:.5f}").background_gradient(cmap="Reds")
        .set_caption(f"Infidelity \u2193 (sum-collapse)   n={n}"))

#7. Sanity Check

In [ ]:
# ── Pre-build cascade models once ──
def reset_layer(model, layer_name):
    layer = getattr(model, layer_name)
    for module in layer.modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            nn.init.kaiming_normal_(module.weight)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)

CASCADE_LAYERS = ["fc", "layer4", "layer3", "layer2", "layer1"]
torch.manual_seed(0)
cascade_models = {}
_base = copy.deepcopy(model); _base.eval()
for layer_name in CASCADE_LAYERS:
    reset_layer(_base, layer_name)
    cascade_models[layer_name] = copy.deepcopy(_base); cascade_models[layer_name].eval()
del _base
print(f"Built {len(cascade_models)} cascade models.")

# ── Pick 10 per named category + 40 from other ──
import random
random.seed(42)
N_PER_CAT  = 10
N_OTHER    = 40
sanity_idx = []
for cat in IMAGE_TYPE_KEYWORDS.keys():       # 6 cats × 10 = 60
    ids = buckets[cat][:]
    random.shuffle(ids)
    sanity_idx.extend(ids[:N_PER_CAT])
other_ids = buckets["other"][:]              # 40 from other = 100 total
random.shuffle(other_ids)
sanity_idx.extend(other_ids[:N_OTHER])
print(f"Running sanity on {len(sanity_idx)} images "
      f"(10 per category × 6 + 40 other = {len(sanity_idx)})")

# ── Sanity cascade loop (cached) ──
import pandas as pd
_cache = CACHE_DIR / "sanity.pkl"
_csv   = CACHE_DIR / "sanity.csv"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        sanity_cascade = pickle.load(f)
    print(f"[cache] Loaded from {_cache}")

elif not FORCE_RECOMPUTE and _csv.exists():
    df_s = pd.read_csv(_csv)
    sanity_cascade = {m: {l: df_s[(df_s.method == m) & (df_s.layer == l)].sort_values("img_i")["rho"].to_list()
                          for l in CASCADE_LAYERS} for m in methods_all}
    print(f"[csv] Reconstructed from {_csv}")

else:
    sanity_cascade = {m: {l: [] for l in CASCADE_LAYERS} for m in methods_all}
    for img_i, idx in enumerate(tqdm(sanity_idx, desc="sanity")):
        r = results[idx]; x, tgt = r["x"], r["target"]

        trained_maps = {}
        for sigma_name in ["adaptive", "σ=0.25"]:
            sf = get_sigma_final(sigma_name, model, x, tgt)
            trained_maps[f"KL-IG ({sigma_name})"] = sum_collapse(raw_klig(model, x, tgt, sf))
        for m, fn in RAW_FN.items():
            trained_maps[m] = sum_collapse(fn(model, x, tgt))

        for layer_name, rand_model in cascade_models.items():
            for sigma_name in ["adaptive", "σ=0.25"]:
                sf  = get_sigma_final(sigma_name, model, x, tgt)
                m   = f"KL-IG ({sigma_name})"
                rand_map = sum_collapse(raw_klig(rand_model, x, tgt, sf))
                sanity_cascade[m][layer_name].append(sanity_similarity(trained_maps[m], rand_map))
            for m, fn in RAW_FN.items():
                rand_map = sum_collapse(fn(rand_model, x, tgt))
                sanity_cascade[m][layer_name].append(sanity_similarity(trained_maps[m], rand_map))

    with open(_cache, "wb") as f:
        pickle.dump(sanity_cascade, f)

    rows = [{"method": m, "layer": l, "img_i": i, "rho": float(v)}
            for m in methods_all for l in CASCADE_LAYERS for i, v in enumerate(sanity_cascade[m][l])]
    pd.DataFrame(rows).to_csv(_csv, index=False)

    pd.DataFrame([{
        "method": m, "layer": l,
        "mean_rho": round(np.mean(sanity_cascade[m][l]), 4),
        "ci95":     round(ci95(sanity_cascade[m][l]), 4),
        "n_images": len(sanity_cascade[m][l]),
    } for m in methods_all for l in CASCADE_LAYERS]).to_csv(CACHE_DIR / "sanity_summary.csv", index=False)

    print(f"[cache] Saved pkl + csv to {CACHE_DIR}")

In [ ]:
# ── Sanity Check: plot from CSV ──
import pandas as pd

df_s = pd.read_csv(CACHE_DIR / "sanity.csv")
df_sum_s = pd.read_csv(CACHE_DIR / "sanity_summary.csv") if (CACHE_DIR / "sanity_summary.csv").exists() else None
n = df_s["img_i"].nunique()

sanity_csv = {m: {l: df_s[(df_s.method==m)&(df_s.layer==l)].sort_values("img_i")["rho"].to_list()
                  for l in CASCADE_LAYERS} for m in methods_all}

fig, ax = plt.subplots(figsize=(11, 5), facecolor="white")
for m in methods_all:
    mu = np.array([np.mean(sanity_csv[m][l]) for l in CASCADE_LAYERS])
    ax.plot(CASCADE_LAYERS, mu, marker="o", label=m, color=COLORS_ALL[m], lw=1.8)
ax.axhline(0, color="gray", ls=":", alpha=0.5)
ax.set_ylabel("Spearman \u03c1 (trained vs randomised)")
ax.set_title(f"Cascading Sanity Check (\u2193 = faithful)   n={n}", fontweight="bold")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

rows = []
for m in methods_all:
    row = {"Method": m}
    for l in CASCADE_LAYERS:
        row[l] = round(np.mean(sanity_csv[m][l]), 3)
    rows.append(row)
df_t = pd.DataFrame(rows).set_index("Method")
display(df_t.style.format("{:.3f}").background_gradient(cmap="RdYlGn_r")
        .set_caption(f"Cascading Sanity Check — Spearman \u03c1 \u2193 (lower = more faithful)  n={n}"))

In [ ]:
# ── Auto-select top 3 images (most distinctive KL-IG attribution) ──
import torch.nn.functional as F
from scipy.ndimage import laplace

def sharpness_score(amap):
    a = amap.detach().cpu().numpy()
    a = gaussian_filter(np.abs(a), sigma=1)
    return np.var(laplace(a))

scores = []
for idx in tqdm(sanity_idx, desc="picking best VIS_IDX"):
    r = results[idx]; x, tgt = r["x"], r["target"]
    sf = get_sigma_final("adaptive", model, x, tgt)
    amap = sum_collapse(raw_klig(model, x, tgt, sf))
    scores.append(sharpness_score(amap))

top3_pos = np.argsort(scores)[-3:][::-1]
TOP3_IDX = [sanity_idx[i] for i in top3_pos]

for rank, idx in enumerate(TOP3_IDX, 1):
    print(f"Top {rank}: idx={idx} | {results[idx]['label_str']} | score={scores[sanity_idx.index(idx)]:.4f}")

  # default to best for downstream cells

In [ ]:
# ── Sanity Check Visual: Original | Trained | Randomised | Corrected (cividis pos) ──
VIS_IDX = TOP3_IDX[2]
RAND_MODEL = cascade_models["layer1"]
r = results[VIS_IDX]; x, tgt = r["x"], r["target"]
img_np = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)

trained_amaps = {}; random_amaps = {}
for sigma_name in ["adaptive", "σ=0.25"]:
    sf = get_sigma_final(sigma_name, model, x, tgt)
    m  = f"KL-IG ({sigma_name})"
    trained_amaps[m] = sum_collapse(raw_klig(model,      x, tgt, sf))
    random_amaps[m]  = sum_collapse(raw_klig(RAND_MODEL, x, tgt, sf))
for m, fn in RAW_FN.items():
    trained_amaps[m] = sum_collapse(fn(model,      x, tgt))
    random_amaps[m]  = sum_collapse(fn(RAND_MODEL, x, tgt))

# Corrected = trained − randomised (model-sensitive component only)
corrected_amaps = {m: trained_amaps[m] - random_amaps[m] for m in methods_all}

def _heatmap_cividis(ax, amap, title=""):
    a = gaussian_filter(np.abs(amap.detach().cpu().numpy()), sigma=2)
    vmax = np.percentile(a, 99) + 1e-12
    ax.imshow(np.clip(a / vmax, 0, 1), cmap="cividis", vmin=0, vmax=1)
    ax.set_title(title, fontsize=7); ax.axis("off")

n = len(methods_all)
fig = plt.figure(figsize=(14, n * 2.0 + 0.5), facecolor="white")
header = [plt.subplot2grid((n+1, 4), (0, c)) for c in range(4)]
header[0].imshow(img_np); header[0].set_title("Original", fontsize=9, fontweight="bold"); header[0].axis("off")
header[1].axis("off"); header[1].set_title("Trained", fontsize=9, fontweight="bold")
header[2].axis("off"); header[2].set_title("Randomised (layer1)", fontsize=9, fontweight="bold")
header[3].axis("off"); header[3].set_title("Corrected (trained − rand)", fontsize=9, fontweight="bold")

for row, m in enumerate(methods_all, start=1):
    ax_l = plt.subplot2grid((n+1, 4), (row, 0))
    ax_l.text(0.5, 0.5, m, ha="center", va="center", transform=ax_l.transAxes, fontsize=8)
    ax_l.axis("off")
    _heatmap_cividis(plt.subplot2grid((n+1, 4), (row, 1)), trained_amaps[m])
    _heatmap_cividis(plt.subplot2grid((n+1, 4), (row, 2)), random_amaps[m])
    _heatmap_cividis(plt.subplot2grid((n+1, 4), (row, 3)), corrected_amaps[m])

plt.suptitle(f"Sanity Check: {r['label_str']}", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.savefig("sanity_visual.png", dpi=150, bbox_inches="tight")
plt.show()

#Images


In [ ]:
from scipy.ndimage import gaussian_filter
import random; random.seed(42)

buckets2 = {cat: [] for cat in IMAGE_TYPE_KEYWORDS}
for idx in eval_idx:
    cat = classify_label(results[idx]["label_str"])
    if cat in buckets2: buckets2[cat].append(idx)

## top 5 highest sigma_final across all categories
all_ids = [idx for ids in buckets2.values() for idx in ids]
VIS = sorted(all_ids,
             key=lambda i: get_sigma_final("adaptive", model,
                                           results[i]["x"], results[i]["target"]),
             reverse=True)[:5]
print([(results[i]["label_str"][:20],
        f"σ_f={get_sigma_final('adaptive', model, results[i]['x'], results[i]['target']):.3f}")
       for i in VIS])

N_STEPS  = 100
N_EPS    = 64
CHECKPTS = [0.25, 0.50, 0.75, 1.00]
N_COLS   = 1 + len(CHECKPTS)

fig, axes = plt.subplots(len(VIS), N_COLS,
                         figsize=(N_COLS * 2.5, len(VIS) * 2.5),
                         facecolor="white")
if len(VIS) == 1: axes = axes[np.newaxis, :]

col_titles = ["original", "0%→25%", "25%→50%", "50%→75%", "75%→100%"]
for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontsize=9, fontweight="bold")

torch.manual_seed(0)
for row, idx in enumerate(tqdm(VIS, desc="path attribution")):
    r  = results[idx]; x, tgt = r["x"], r["target"]
    sf = get_sigma_final("adaptive", model, x, tgt)

    img_np = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_ylabel(f"{r['label_str'][:16]}\nσ_f={sf:.2f}", fontsize=7, labelpad=4)

    snap_steps = {int(c * N_STEPS): c for c in CHECKPTS}
    snap_maps  = {}
    prev       = torch.zeros(x.shape[1], *x.shape[-2:], device=x.device)
    cumulative = torch.zeros(x.shape[1], *x.shape[-2:], device=x.device)

    for step in range(1, N_STEPS + 1):
        t     = step / N_STEPS
        eps_b = torch.randn(N_EPS, *x.shape[1:], device=x.device)
        x_t   = (x + eps_b * t * sf).detach().requires_grad_(True)
        model(x_t)[:, tgt].sum().backward()
        cumulative += (x_t.grad * (-eps_b * sf)).mean(0).detach() / N_STEPS
        if step in snap_steps:
            snap_maps[snap_steps[step]] = (cumulative - prev).clone()
            prev = cumulative.clone()

    for ci, c in enumerate(CHECKPTS, start=1):
        a    = snap_maps[c].clamp(min=0).sum(0).detach().cpu().numpy()
        vmax = np.percentile(a, 99) + 1e-8   # per-bin normalization
        img  = plt.get_cmap("cividis")(np.clip(a / vmax, 0, 1))[:, :, :3]
        axes[row, ci].imshow(img)
        axes[row, ci].axis("off")

    axes[row, 0].axis("off")

plt.suptitle("KL-IG — attribution \n",
             fontweight="bold", fontsize=12)
plt.tight_layout(pad=0.3)
plt.savefig("klig_path_grid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

N_PREVIEW  = 30
CONF_GAP   = 0.20
MIN_TOP1   = 0.25     # model must be at least 25% confident

candidates = []
for idx in eval_idx:
    r = results[idx]; x, tgt = r["x"], r["target"]
    with torch.no_grad():
        probs    = model(x).softmax(-1)[0]
        top_p, top_c = probs.topk(3)
        top_p    = top_p.cpu().numpy(); top_c = top_c.cpu().tolist()
    if top_p[0] < MIN_TOP1: continue
    gap = float(top_p[0] - top_p[1])
    if gap < CONF_GAP:
        candidates.append({"idx": idx, "label": r["label_str"],
                           "target": tgt, "top_p": top_p,
                           "top_c": top_c, "gap": gap})

candidates.sort(key=lambda d: d["gap"])
candidates = candidates[:N_PREVIEW]

# preview grid
NCOLS = 5
NROWS = math.ceil(len(candidates) / NCOLS)
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS * 2.5, NROWS * 3.0))
axes = axes.ravel()

for i, d in enumerate(candidates):
    img_np = np.clip(denormalize(results[d["idx"]]["x"][0])
                     .detach().cpu().permute(1,2,0).numpy(), 0, 1)
    axes[i].imshow(img_np); axes[i].axis("off")
    axes[i].set_title(
        f"[{i}] {d['label'][:14]}\n"
        f"{imagenet_labels[d['top_c'][0]][:14]} {d['top_p'][0]:.0%}\n"
        f"{imagenet_labels[d['top_c'][1]][:14]} {d['top_p'][1]:.0%}",
        fontsize=6, color="black")

for ax in axes[len(candidates):]: ax.axis("off")
plt.suptitle("Confused candidates — note the index [i] and set VIS_CONFUSED below",
             fontsize=10, fontweight="bold")
plt.tight_layout(pad=0.3); plt.show()

# ── then pick by index ──
# VIS_CONFUSED = [candidates[i] for i in [0, 3, 7, 11, 15]]

In [ ]:
from scipy.ndimage import gaussian_filter

VIS_CONFUSED = [candidates[i] for i in [7, 9, 13, 14, 23]]
TOP_K = 3

def _saliency(model, x, cls, sf):
    raw  = raw_klig(model, x, cls, sf)
    a    = raw.clamp(min=0).sum(dim=0).detach().cpu().numpy()
    vmax = np.percentile(a, 99); vmax = vmax if vmax > 1e-12 else 1.0
    return plt.get_cmap("cividis")(np.clip(a / vmax, 0, 1))[:, :, :3]

N_COLS = 1 + TOP_K
fig, axes = plt.subplots(len(VIS_CONFUSED), N_COLS,
                         figsize=(N_COLS * 3.0, len(VIS_CONFUSED) * 3.0),
                         facecolor="white")

for row, d in enumerate(VIS_CONFUSED):
    r  = results[d["idx"]]; x = r["x"]
    sf = get_sigma_final("adaptive", model, x, r["target"])

    img_np = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    axes[row, 0].imshow(img_np); axes[row, 0].axis("off")
    axes[row, 0].set_ylabel(d["label"][:18], fontsize=8)
    if row == 0:
        axes[row, 0].set_title("Original", fontsize=10, fontweight="bold")

    for ci, (cls, prob) in enumerate(zip(d["top_c"], d["top_p"]), start=1):
        sal   = _saliency(model, x, cls, sf)
        match = "✓" if cls == d["target"] else "✗"
        axes[row, ci].imshow(sal); axes[row, ci].axis("off")
        axes[row, ci].set_title(
            f"{match} {imagenet_labels[cls][:18]}\nP={prob:.3f}",
            fontsize=9, fontweight="bold",
            color="green" if match == "✓" else "red")

plt.suptitle("Saliency maps for competing predictions", fontsize=12, fontweight="bold")
plt.tight_layout(pad=0.4)
plt.savefig("class_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

import random
random.seed(42)

IMAGE_TYPE_KEYWORDS = {
    "animal":    ["dog","cat","bird","fish","snake","lizard","frog","elephant",
                  "tiger","lion","bear","fox","wolf","deer","horse","cow","sheep",
                  "monkey","ape","panda","leopard","zebra","penguin","owl","eagle",
                  "whale","shark","spider","beetle","rabbit","squirrel","goat",
                  "pig","camel","kangaroo","ostrich"],
    "vehicle":   ["car","truck","bus","train","boat","ship","airplane","bicycle",
                  "motorcycle","cab","wagon","trailer","scooter","ambulance"],
  #  "object":    ["chair","table","lamp","bottle","cup","book","keyboard","phone",
  #                "computer","clock","watch","monitor","camera","scissors",
  #                "umbrella","mask","helmet","backpack","tie"],
    "food":      ["bread","pizza","cake","apple","banana","orange","broccoli",
                  "sandwich","hotdog","donut","cheese","mushroom","strawberry",
                  "lemon","ice cream"]
   # "nature":    ["tree","flower","mountain","beach","lake","forest","valley",
    #              "cliff","seashore","volcano","coral","alp","hill","river"],
  #  "structure": ["building","castle","church","tower","bridge","dome","barn",
   #               "lighthouse","palace","fountain","stadium"],
}

def classify_label(label_str):
    s = label_str.lower()
    for cat, kws in IMAGE_TYPE_KEYWORDS.items():
        if any(kw in s for kw in kws): return cat
    return "other"

buckets = {cat: [] for cat in IMAGE_TYPE_KEYWORDS}; buckets["other"] = []
for idx in eval_idx:
    buckets[classify_label(results[idx]["label_str"])].append(idx)

N_PER_CATEGORY = 1
vis_idx, vis_cats = [], []
for cat in IMAGE_TYPE_KEYWORDS.keys():
    ids = buckets[cat][:]; random.shuffle(ids)
    for i in ids[:N_PER_CATEGORY]:
        vis_idx.append(i); vis_cats.append(cat)

print(f"Selected {len(vis_idx)} images across {len(set(vis_cats))} categories: {sorted(set(vis_cats))}")

KLIG_SIGMAS_VIS = ["adaptive", "σ=0.25"]
BASELINE_MS_VIS = ["IDG", "ExpGrad", "IG-zero", "Blur-IG", "SmoothGrad", "Vanilla Grad"]
N_VIS  = len(vis_idx)
N_COLS = 1 + len(KLIG_SIGMAS_VIS) + len(BASELINE_MS_VIS)

fig, axes = plt.subplots(N_VIS * 2, N_COLS,
                         figsize=(N_COLS * 2.0, N_VIS * 4.0),
                         facecolor="white")
if N_VIS == 1: axes = axes[np.newaxis, :]

def _pos_neg_pair(ax_pos, ax_neg, x, raw_attr):
    for ax, a_hw, cmap in [
        (ax_pos, raw_attr.clamp(min=0).sum(dim=0),       "cividis"),
        (ax_neg, raw_attr.clamp(max=0).abs().sum(dim=0), "cividis_r"),
    ]:
        a = a_hw.detach().cpu().numpy()
        vmax = np.percentile(a, 99); vmax = vmax if vmax > 1e-12 else 1.0
        ax.imshow(np.clip(a / vmax, 0, 1), cmap=cmap, vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])

for row, (idx, cat) in enumerate(tqdm(list(zip(vis_idx, vis_cats)),
                                       desc="qual all methods")):
    r = results[idx]; x, tgt = r["x"], r["target"]
    img_np = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    r_pos, r_neg = row * 2, row * 2 + 1
    for ri in [r_pos, r_neg]:
        axes[ri, 0].imshow(img_np); axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    axes[r_pos, 0].set_ylabel(f"[{cat}]\n{r['label_str'][:18]}\n(+)",
                               fontsize=7, rotation=0, labelpad=65, va="center", fontweight="bold")
    axes[r_neg, 0].set_ylabel("(−)", fontsize=9, rotation=0, labelpad=65, va="center")

    for col_offset, sigma_name in enumerate(KLIG_SIGMAS_VIS):
        sf = get_sigma_final(sigma_name, model, x, tgt)
        raw = raw_klig(model, x, tgt, sf)
        _pos_neg_pair(axes[r_pos, 1 + col_offset], axes[r_neg, 1 + col_offset], x, raw)

    for col_offset, m in enumerate(BASELINE_MS_VIS):
        raw = RAW_FN[m](model, x, tgt)
        _pos_neg_pair(axes[r_pos, 1 + len(KLIG_SIGMAS_VIS) + col_offset],
                      axes[r_neg, 1 + len(KLIG_SIGMAS_VIS) + col_offset], x, raw)

axes[0, 0].set_title("Original", fontsize=9, fontweight="bold")
col_titles = [f"KL-IG ({sn})" for sn in KLIG_SIGMAS_VIS] + BASELINE_MS_VIS
for col, ttl in enumerate(col_titles, start=1):
    axes[0, col].set_title(ttl, fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig("qualitative_all_methods_by_type.png", dpi=180, bbox_inches="tight")
plt.show()

#Class Sensitivity


In [ ]:
# ── WordNet Scatter — Fixed: signed attr + per-class sigma ─────────────────
#import random, pickle
#random.seed(42)
#
#N_IMGS    = 15
#N_PER_BIN = 4
#
#DIST_BINS = [(1, 4), (5, 8), (9, 12), (13, 16), (17, 25)]
#
#ATTR_FNS = {
#    "KL-IG (adaptive)": lambda x, cls, sf: raw_klig(model, x, cls, sf),
#    "KL-IG (σ=0.25)":   lambda x, cls, sf: raw_klig(model, x, cls, 0.25),
#    "IDG":              lambda x, cls, sf: raw_idg(model, x, cls),
#    "ExpGrad":          lambda x, cls, sf: raw_expgrad(model, x, cls),
#    "IG-zero":          lambda x, cls, sf: raw_ig_zero(model, x, cls),
#    "Blur-IG":          lambda x, cls, sf: raw_blur_ig(model, x, cls),
#    "SmoothGrad":       lambda x, cls, sf: raw_smoothgrad(model, x, cls),
#    "Vanilla Grad":     lambda x, cls, sf: raw_vanilla(model, x, cls),
#}
#
#if "cls_synsets" not in dir():
#    cls_synsets = {i: label_to_synset(imagenet_labels[i]) for i in range(1000)}
#
#def get_stratified_classes(true_cls):
#    s_true = cls_synsets.get(true_cls)
#    if s_true is None: return []
#    buckets = {b: [] for b in DIST_BINS}
#    for ci in range(1000):
#        if ci == true_cls: continue
#        s_c = cls_synsets.get(ci)
#        if s_c is None: continue
#        d = s_true.shortest_path_distance(s_c)
#        if d is None: continue
#        for (lo, hi) in DIST_BINS:
#            if lo <= d <= hi:
#                buckets[(lo, hi)].append((ci, d)); break
#    result = []
#    for b, items in buckets.items():
#        result.extend(random.sample(items, min(N_PER_BIN, len(items))))
#    return result
#
#def get_sf_for_class(model, x, ci, sf_true, min_sf=0.10):
#    """Per-class sigma for KL-IG: floor to min_sf if model has no confidence in ci."""
#    with torch.no_grad():
#        p_ci = float(model(x).softmax(-1)[0, ci])
#    if p_ci < 0.05:
#        return max(sf_true, min_sf)
#    return get_sigma_final("adaptive", model, x, ci)
#
#_cache = CACHE_DIR / "wn_scatter_fixed.pkl"
#if _cache.exists():
#    scatter_all = pickle.load(open(_cache, "rb"))
#    print(f"[resume] loaded {sum(len(v) for v in scatter_all.values())} existing points")
#else:
#    scatter_all = {m: [] for m in ATTR_FNS}
#
#imgs_done = 0
#seen_idx  = set()
#
#for idx in tqdm(eval_idx[:60], desc="wn-scatter-fixed"):
#    if imgs_done >= N_IMGS: break
#    if idx in seen_idx: continue
#    r = results[idx]; x, tgt = r["x"], r["target"]
#    with torch.no_grad():
#        probs = model(x).softmax(-1)[0]
#    if int(probs.argmax()) != tgt or float(probs[tgt]) < 0.40: continue
#    if cls_synsets.get(tgt) is None: continue
#
#    sf_true  = get_sigma_final("adaptive", model, x, tgt)
#    comp_cls = get_stratified_classes(tgt)
#    if not comp_cls: continue
#
#    # FIX 2: signed — preserves negative attribution signal
#    attr_true = {m: fn(x, tgt, sf_true).sum(0).detach().cpu().numpy().ravel()
#                 for m, fn in ATTR_FNS.items()}
#
#    for ci, dist in comp_cls:
#        # FIX 1: per-class sigma for KL-IG; others ignore sf anyway
#        sf_ci = get_sf_for_class(model, x, ci, sf_true)
#
#        for m, fn in ATTR_FNS.items():
#            sf_use = sf_ci if "KL-IG" in m else sf_true
#            a_c    = fn(x, ci, sf_use).sum(0).detach().cpu().numpy().ravel()
#            rho, _ = spearmanr(attr_true[m], a_c)
#            scatter_all[m].append((dist, 1.0 - float(rho)))
#
#    seen_idx.add(idx)
#    imgs_done += 1
#    pickle.dump(scatter_all, open(_cache, "wb"))
#    print(f"  [{imgs_done}/{N_IMGS}] {r['label_str'][:28]}  +{len(comp_cls)} pairs  [saved]")
#
## ── Plot ──
#fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor="white")
#axes = axes.ravel()
#
#for ax, mname in zip(axes, ATTR_FNS):
#    pts   = scatter_all[mname]
#    dists = np.array([p[0] for p in pts])
#    divs  = np.array([p[1] for p in pts])
#    jitter = np.random.default_rng(0).uniform(-0.18, 0.18, size=len(dists))
#    ax.scatter(dists + jitter, divs, alpha=0.35, s=18,
#               c=dists, cmap="plasma", edgecolors="none")
#    for d in sorted(set(dists)):
#        vals = divs[dists == d]
#        if len(vals) >= 2:
#            ax.errorbar(d, vals.mean(), yerr=vals.std(), fmt="o",
#                        color="black", ms=4, lw=1.2, zorder=5)
#    z  = np.polyfit(dists, divs, 1)
#    xs = np.linspace(dists.min(), dists.max(), 100)
#    ax.plot(xs, np.poly1d(z)(xs), "r--", lw=1.5)
#    rho_m, pval = spearmanr(dists, divs)
#    sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))
#    ax.set_title(f"{mname}\nρ = {rho_m:.3f}  {sig}", fontsize=9, fontweight="bold")
#    ax.set_xlabel("WN dist", fontsize=8)
#    ax.set_ylabel("divergence", fontsize=8)
#    ax.grid(True, alpha=0.3, linestyle="--")
#
#plt.suptitle("Semantic Distance vs Attribution Divergence\nFixed: signed attr + per-class sigma",
#             fontsize=13, fontweight="bold")
#plt.tight_layout(rect=[0, 0, 1, 0.97])
#plt.savefig("wordnet_scatter_fixed.png", dpi=150, bbox_inches="tight")
#plt.show()
#
#print(f"\n{'Method':<22} {'rho':>8} {'p':>10} {'n':>6}")
#print("-" * 50)
#for m, pts in scatter_all.items():
#    dists = np.array([p[0] for p in pts])
#    divs  = np.array([p[1] for p in pts])
#    rho_m, pval = spearmanr(dists, divs)
#    sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))
#    print(f"  {m:<20} {rho_m:>8.3f} {pval:>10.4f} {len(pts):>6}  {sig}")

In [ ]:
# ── Class Sensitivity: End-to-End — All Methods ────────────────────────────
import pickle, itertools, nltk
nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr
from torchvision.models import ResNet50_Weights

# ── Setup ──
PROB_THRESH = 0.1
N_IMGS      = 1000

ATTR_FNS = {
    "KL-IG (adaptive)": lambda x, cls, sf: raw_klig(model, x, cls, sf),
    "KL-IG (σ=0.25)":   lambda x, cls, sf: raw_klig(model, x, cls, 0.25),
    "IDG":              lambda x, cls, sf: raw_idg(model, x, cls),
    "ExpGrad":          lambda x, cls, sf: raw_expgrad(model, x, cls),
    "IG-zero":          lambda x, cls, sf: raw_ig_zero(model, x, cls),
    "Blur-IG":          lambda x, cls, sf: raw_blur_ig(model, x, cls),
    "SmoothGrad":       lambda x, cls, sf: raw_smoothgrad(model, x, cls),
    "Vanilla Grad":     lambda x, cls, sf: raw_vanilla(model, x, cls),
}

imagenet_labels = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

def label_to_synset(label):
    phrase = label.lower().split(",")[0].strip()
    candidates = []
    for candidate in [phrase.replace(" ", "_"), phrase]:
        candidates.extend(wn.synsets(candidate, pos=wn.NOUN))
    if not candidates:
        for w in sorted(phrase.split(), key=len, reverse=True):
            if len(w) > 3:
                candidates.extend(wn.synsets(w, pos=wn.NOUN))
                if candidates: break
    if not candidates:
        return None
    return max(candidates, key=lambda s: s.min_depth())

cls_synsets = {i: label_to_synset(imagenet_labels[i]) for i in range(1000)}

# ── Step 1: filter images with ≥2 classes above PROB_THRESH (cached) ──
_cache_imgs = CACHE_DIR / "multiprob_imgs.pkl"

if _cache_imgs.exists():
    multi_imgs = pickle.load(open(_cache_imgs, "rb"))
    print(f"[cache] loaded {len(multi_imgs)} multi-class images")
else:
    multi_imgs = []
    for idx in eval_idx[:N_IMGS]:
        r = results[idx]; x = r["x"]
        with torch.no_grad():
            probs = model(x).softmax(-1)[0].cpu()
        high = (probs > PROB_THRESH).nonzero(as_tuple=True)[0].tolist()
        if len(high) < 2: continue
        high = sorted(high, key=lambda c: probs[c].item(), reverse=True)
        multi_imgs.append({"idx": idx, "x": x,
                           "high_cls":   high,
                           "high_probs": [probs[c].item() for c in high]})
    pickle.dump(multi_imgs, open(_cache_imgs, "wb"))
    print(f"[computed] {len(multi_imgs)} multi-class images — saved to cache")

n_pairs_total = sum(len(list(itertools.combinations(d["high_cls"], 2))) for d in multi_imgs)
print(f"Step 1 — {len(multi_imgs)} images with ≥2 classes > {PROB_THRESH}  "
      f"({n_pairs_total} pairs total)")

# ── Step 2: attribution divergence + incremental save ──
_cache   = CACHE_DIR / "multiprob_e2e.pkl"
_cache_n = CACHE_DIR / "multiprob_e2e_n.pkl"

if _cache.exists():
    scatter_all = pickle.load(open(_cache, "rb"))
    done_imgs   = pickle.load(open(_cache_n, "rb"))
    print(f"[resume] {sum(len(v) for v in scatter_all.values())} points, img {done_imgs}/{len(multi_imgs)}")
else:
    scatter_all = {m: [] for m in ATTR_FNS}
    done_imgs   = 0

for d in tqdm(multi_imgs[done_imgs:], desc="all methods"):
    x        = d["x"]
    high_cls = d["high_cls"]
    top_cls  = high_cls[0]
    sf       = get_sigma_final("adaptive", model, x, top_cls)

    for ci, cj in itertools.combinations(high_cls, 2):
        s_i = cls_synsets.get(ci)
        s_j = cls_synsets.get(cj)
        if s_i is None or s_j is None: continue
        dist = s_i.shortest_path_distance(s_j)
        if dist is None: continue

        for mname, fn in ATTR_FNS.items():
            a_i    = fn(x, ci, sf).clamp(min=0).sum(0).detach().cpu().numpy().ravel()
            a_j    = fn(x, cj, sf).clamp(min=0).sum(0).detach().cpu().numpy().ravel()
            rho, _ = spearmanr(a_i, a_j)
            scatter_all[mname].append((dist, 1.0 - float(rho)))

    done_imgs += 1
    pickle.dump(scatter_all, open(_cache,   "wb"))
    pickle.dump(done_imgs,   open(_cache_n, "wb"))

print(f"Step 2 — {len(scatter_all['KL-IG (adaptive)'])} pairs per method")

# ── Step 3: 2×4 subplot ──
fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor="white")
axes = axes.ravel()

for ax, mname in zip(axes, ATTR_FNS):
    pts   = scatter_all[mname]
    dists = np.array([p[0] for p in pts])
    divs  = np.array([p[1] for p in pts])
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(dists))
    ax.scatter(dists + jitter, divs, alpha=0.35, s=15,
               c=dists, cmap="plasma", edgecolors="none")
    for d in sorted(set(dists)):
        vals = divs[dists == d]
        if len(vals) >= 2:
            ax.errorbar(d, vals.mean(), yerr=vals.std(), fmt="o",
                        color="black", ms=4, lw=1.2, zorder=5)
        else:
            ax.scatter([d], [vals.mean()], color="black", s=30, zorder=5)
    z  = np.polyfit(dists, divs, 1)
    xs = np.linspace(dists.min(), dists.max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), "r--", lw=1.5)
    rho_m, pval = spearmanr(dists, divs)
    ax.set_title(f"{mname}\nρ = {rho_m:.3f}", fontsize=9, fontweight="bold")
    ax.set_xlabel("WN dist", fontsize=8)
    ax.set_ylabel("divergence", fontsize=8)
    ax.grid(True, alpha=0.3, linestyle="--")

plt.suptitle("Class Sensitivity: Attribution divergence VS Wordnet Distance",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("multiprob_scatter_all.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'Method':<22} {'rho':>8} {'p':>10} {'n':>6}")
print("-" * 50)
for mname, pts in scatter_all.items():
    dists = np.array([p[0] for p in pts])
    divs  = np.array([p[1] for p in pts])
    rho_m, pval = spearmanr(dists, divs)
    print(f"  {mname:<20} {rho_m:>8.3f} {pval:>10.4f} {len(pts):>6}")

In [ ]:
import random
N = 2
seen = set()

shuffled = all_candidates.copy()
random.shuffle(shuffled)

def get_unique(candidates, dist_min, dist_max):
    out = []
    for dist, d, ci, cj in candidates:
        if not (dist_min <= dist <= dist_max): continue
        if results[d["idx"]]["target"] != d["high_cls"][0]: continue
        key = frozenset([ci, cj])
        if key in seen: continue
        seen.add(key)
        out.append((d, ci, cj, dist))
        if len(out) == N: break
    return out

low_pairs  = get_unique(shuffled,  0,  8)
med_pairs  = get_unique(shuffled,  9, 16)
high_pairs = get_unique(shuffled, 17, 99)

_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
_STD  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

BUCKET_COLOR = {"Low": "navy", "Med": "darkgreen", "High": "darkred"}
all_pairs = [("Low", p) for p in low_pairs] + \
            [("Med", p) for p in med_pairs] + \
            [("High", p) for p in high_pairs]

fig, axes = plt.subplots(len(all_pairs), 3, figsize=(13, 4.2 * len(all_pairs)), facecolor="white")

for row, (bucket, (d, ci, cj, dist)) in enumerate(all_pairs):
    x   = d["x"]
    sf  = get_sigma_final("adaptive", model, x, d["high_cls"][0])
    a_i = raw_klig(model, x, ci, sf).clamp(min=0).sum(0).detach().cpu().numpy()
    a_j = raw_klig(model, x, cj, sf).clamp(min=0).sum(0).detach().cpu().numpy()
    rho, _ = spearmanr(a_i.ravel(), a_j.ravel())
    img_show = (x[0].cpu() * _STD + _MEAN).clamp(0, 1).permute(1, 2, 0).detach().numpy()
    p_i = d["high_probs"][d["high_cls"].index(ci)]
    p_j = d["high_probs"][d["high_cls"].index(cj)]

    axes[row, 0].imshow(img_show)
    axes[row, 0].set_title(f"[{bucket}]  WN dist={dist}  div={1-rho:.3f}",
                           fontsize=9, fontweight="bold", color=BUCKET_COLOR[bucket])
    axes[row, 0].axis("off")

    vmax = np.percentile(np.concatenate([a_i.ravel(), a_j.ravel()]), 99)
    axes[row, 1].imshow(a_i, cmap="cividis", vmin=0, vmax=vmax)
    axes[row, 1].set_title(f"{imagenet_labels[ci]}\n(p={p_i:.2f})", fontsize=8)
    axes[row, 1].axis("off")
    axes[row, 2].imshow(a_j, cmap="cividis", vmin=0, vmax=vmax)
    axes[row, 2].set_title(f"{imagenet_labels[cj]}\n(p={p_j:.2f})", fontsize=8)
    axes[row, 2].axis("off")

plt.suptitle("Class Sensitivity Attribution", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("wn_viz.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── KL-IG Path Visualization: Attribution Segments (heatmap only) ──────────
import math, pickle

SEGMENTS   = [(0.0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0)]
SEG_LABELS = ["σ(0, 0.25)", "σ(0.25, 0.5)", "σ(0.5, 0.75)", "σ(0.75, 1)"]
N_STEPS    = 50

_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
_STD  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

BLOCKLIST = {"missile", "projectile", "warplane", "cannon", "rifle", "revolver",
             "assault rifle", "shotgun", "bow"}

def compute_partial_klig(model, x, target, sigma_final):
    x1           = x.squeeze(0)
    mu_final     = x1.detach()
    logvar_final = torch.full_like(mu_final, 2 * math.log(sigma_final))
    steps        = torch.linspace(0.5/N_STEPS, 1 - 0.5/N_STEPS, N_STEPS)
    seg_sums     = [torch.zeros_like(mu_final) for _ in SEGMENTS]
    model.eval()
    saved = [(p, p.requires_grad) for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    for t_val in steps:
        t        = float(t_val)
        mu_t     = (t * mu_final).detach().requires_grad_(True)
        logvar_t = (t * logvar_final).detach().requires_grad_(True)
        eps      = torch.randn(N_SAMPLES, *x1.shape, device=x1.device)
        x_samp   = mu_t.unsqueeze(0) + (0.5 * logvar_t).exp().unsqueeze(0) * eps
        obj      = model(x_samp)[:, target].mean()
        obj.backward()
        with torch.no_grad():
            g = mu_t.grad * mu_final + logvar_t.grad * logvar_final
        for i, (a, b) in enumerate(SEGMENTS):
            if a <= t < b or (b == 1.0 and t >= a):
                seg_sums[i] += g
    for p, rg in saved: p.requires_grad_(rg)
    return [s / N_STEPS for s in seg_sums]

# ── Candidates from full 1k eval set (cached) ──
_cand_cache = CACHE_DIR / "path_viz_candidates.pkl"

if _cand_cache.exists():
    candidates = pickle.load(open(_cand_cache, "rb"))
    print(f"[cache] {len(candidates)} candidates")
else:
    candidates = []
    for idx in tqdm(eval_idx[:1000], desc="scanning"):
        r = results[idx]
        x, tgt = r["x"], r["target"]
        with torch.no_grad():
            top1 = model(x).softmax(-1)[0].argmax().item()
        if top1 != tgt: continue
        if imagenet_labels[tgt].lower() in BLOCKLIST: continue
        sf = get_sigma_final("adaptive", model, x, tgt)
        candidates.append(({"idx": idx, "x": x, "high_cls": [tgt]}, sf))
    candidates.sort(key=lambda x: x[1])
    pickle.dump(candidates, open(_cand_cache, "wb"))
    print(f"[computed] {len(candidates)} candidates")

print(f"σ range: {candidates[0][1]:.3f} – {candidates[-1][1]:.3f}")
indices  = np.linspace(0, len(candidates) - 1, 6, dtype=int)
selected = [candidates[i] for i in indices]

# ── Plot ──
fig, axes = plt.subplots(6, 5, figsize=(4.2 * 5, 4 * 6), facecolor="white")

axes[0, 0].set_title("Original", fontsize=9, fontweight="bold")
for col, lbl in enumerate(SEG_LABELS):
    axes[0, col + 1].set_title(lbl, fontsize=9, fontweight="bold")

for row, (d, sf) in enumerate(tqdm(selected, desc="images")):
    x      = d["x"]
    target = d["high_cls"][0]
    label  = imagenet_labels[target]

    img_show = (x[0].cpu() * _STD + _MEAN).clamp(0, 1).permute(1, 2, 0).detach().numpy()
    axes[row, 0].imshow(img_show)
    axes[row, 0].set_xlabel(f"{label}\nσ_final = {sf:.3f}", fontsize=8, labelpad=4)
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    for spine in axes[row, 0].spines.values(): spine.set_visible(False)

    partial_attrs = compute_partial_klig(model, x, target, sf)
    vmax = np.percentile(np.concatenate([
        a.clamp(min=0).sum(0).detach().cpu().numpy().ravel()
        for a in partial_attrs]), 99)

    for col, ((a, b), attr) in enumerate(zip(SEGMENTS, partial_attrs)):
        attr_map = attr.clamp(min=0).sum(0).detach().cpu().numpy()
        axes[row, col + 1].imshow(attr_map, cmap="cividis", vmin=0, vmax=vmax)
        axes[row, col + 1].axis("off")

plt.suptitle("KL-IG: Attribution along Integration Path",
             fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("klig_path_segments.png", dpi=150, bbox_inches="tight")
plt.show()